In [ ]:
# ==== REGION 2/3: native-resolution corpus of the studies in chunks, then the specialists ====
import glob as _rs_glob, json as _rs_json, os as _rs_os, shutil as _rs_sh, subprocess as _rs_sp, sys as _rs_sys, time as _rs_time
import numpy as _rs_np, pandas as _rs_pd
_RS_SPEC = {'side': ['raptor_rs_side_last.pt', 'raptor_rs_side.pt'], 'regions': [{'findings': ['PF OA'], 'rounds': [['raptor_rs_rs3pf0.pt'], ['raptor_rs_rs3pf1.pt']]}], 'chunk': 100, 'passes': 1, 'channels_last': False}
_RS_LAB = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_RS_STRICT = True     # True (alone / parity): a failure stops the notebook; False: the base stands
_RS_W = "/kaggle/working/_rs"; _RS_C = "/tmp/rs_corpus"; _RS_CSV = "/kaggle/working/_rs_pred.csv"
_RS_NB_T0 = globals().get("_RS_NB_T0", _rs_time.time())     # notebook start
_rs_ok = False


def _rs_find(name):   # weights only: never walk the competition's DICOM tree
    for d in sorted(_rs_glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower() or "abnormality-detection" in d:
            continue
        hits = _rs_glob.glob(d + "**/" + name, recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(name + " not found under /kaggle/input")


def _rs_run(cmd, env=None, tail=12):
    r = _rs_sp.run(cmd, env=dict(_rs_os.environ, **(env or {})), stdout=_rs_sp.PIPE, stderr=_rs_sp.STDOUT, text=True)
    print("\n".join(r.stdout.splitlines()[-tail:]), flush=True)
    if r.returncode != 0:
        raise RuntimeError(f"{_rs_os.path.basename(cmd[1])} rc={r.returncode}")


def _rs_test_ids():
    if []:
        return [str(u) for u in []]
    if _rs_os.path.exists("/kaggle/working/submission.csv"):     # the base's own list of studies
        return _rs_pd.read_csv("/kaggle/working/submission.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
    for d, _, f in _rs_os.walk("/kaggle/input"):
        if "test.csv" in f and "test_series.csv" in f:
            return _rs_pd.read_csv(d + "/test.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
    raise RuntimeError("test.csv not found")


def _rs_score(ids, stop_at):
    # Rounds of checkpoints over chunks of studies. Round 0 must fit before `stop_at` (seconds of notebook time,
    # projected from its first chunk); a later round is started only if it is projected to end before it.
    regs, ch = _RS_SPEC["regions"], _RS_SPEC["chunk"]
    chunks = [ids[i:i + ch] for i in range(0, len(ids), ch)]
    base = [_rs_sys.executable, _RS_W + "/region_spec.py", "--labels", "none", "--region", "tf", "--data_root", _RS_C,
            "--n_shards", "1", "--out", _RS_W, "--workers", "3", "--bs", "6"]
    env = {"RS_CORPUS": _RS_C, "RS_SPLIT": 'test', "RS_IDS": _RS_W + "/ids.json"}
    tot, cnt, info = {}, {}, {"rounds": 0, "errors": 0, "empty": 0}
    for rd in range(max(len(r["rounds"]) for r in regs)):
        cost = sum(len(r["rounds"][rd]) for r in regs if len(r["rounds"]) > rd)
        if rd > 0:
            need = 1.08 * t_r0 * cost / cost0
            if _rs_time.time() - _RS_NB_T0 + need > stop_at:
                print(f"[region 2/3] round {rd} NOT started: needs ~{need / 3600:.2f} h, notebook at "
                      f"{(_rs_time.time() - _RS_NB_T0) / 3600:.2f} h of {stop_at / 3600:.1f} h", flush=True)
                break
        part, t_rd = {}, _rs_time.time()
        for ci, chunk in enumerate(chunks):
            tc = _rs_time.time()
            _rs_sh.rmtree(_RS_C, ignore_errors=True); _rs_os.makedirs(_RS_C)
            _rs_json.dump(chunk, open(_RS_W + "/ids.json", "w"))
            _rs_run([_rs_sys.executable, _RS_W + "/dense512_build.py"], env, tail=1)
            meta = _rs_json.load(open(_RS_C + "/meta_s0.json"))
            assert meta["done"] == len(chunk), f"corpus built {meta['done']} of {len(chunk)} studies"
            if rd == 0:
                info["errors"] += len(meta["errors"]); info["empty"] += meta["empty_studies"]
            side = []
            if _RS_SPEC["side"]:
                sf = _RS_W + f"/side_{ci}.npz"       # one side call a chunk, kept for the later rounds
                if not _rs_os.path.exists(sf):
                    _rs_run(base + ["--predict", ",".join(_rs_find(c) for c in _RS_SPEC["side"]), "--tag", "side",
                                    "--eval_passes", "1"], tail=2)
                    _rs_os.replace(_RS_W + "/raptor_side_side.npz", sf)
                    zs = _rs_np.load(sf, allow_pickle=True)
                    info.setdefault("side", {}).update(zip(zs["ids"].astype(str).tolist(), zs["p_left"].astype(float).tolist()))
                side = ["--side_file", sf]
            for i, r in enumerate(regs):
                if len(r["rounds"]) <= rd:
                    continue
                _rs_run(base + ["--predict", ",".join(_rs_find(c) for c in r["rounds"][rd]), "--tag", f"r{i}",
                                "--eval_passes", str(_RS_SPEC["passes"]), "--only_findings", ";".join(r["findings"])]
                        + (["--channels_last"] if _RS_SPEC.get("channels_last") else []) + side, tail=2)
                z = _rs_np.load(_RS_W + f"/raptor_rspred_r{i}.npz", allow_pickle=True)
                assert z["ids"].astype(str).tolist() == chunk, "the specialists' studies are not the chunk's"
                for f in r["findings"]:
                    part.setdefault(f, []).append(_rs_pd.Series(z["pred"][:, _RS_LAB.index(f)].astype(float), index=chunk) * len(r["rounds"][rd]))
                    if ci == 0:
                        cnt[f] = cnt.get(f, 0) + len(r["rounds"][rd])
            _rs_sh.rmtree(_RS_C, ignore_errors=True)
            el = _rs_time.time() - t_rd
            print(f"[region 2/3] round {rd} chunk {ci + 1}/{len(chunks)} | {len(chunk)} studies in {_rs_time.time() - tc:.0f}s | "
                  f"round eta {el / (ci + 1) * (len(chunks) - ci - 1) / 60:.0f} min | notebook at {(_rs_time.time() - _RS_NB_T0) / 3600:.2f} h", flush=True)
            if rd == 0 and ci == 0 and len(chunks) > 1:
                proj = (_rs_time.time() - tc) * len(ids) / len(chunk)
                if tc - _RS_NB_T0 + proj > stop_at:
                    raise RuntimeError(f"no time left: round 0 needs ~{proj / 3600:.2f} h from {(tc - _RS_NB_T0) / 3600:.2f} h, limit {stop_at / 3600:.1f} h")
        if rd == 0:
            t_r0, cost0 = _rs_time.time() - t_rd, cost
        for f, ps in part.items():
            tot[f] = tot.get(f, 0.0) + _rs_pd.concat(ps)
        info["rounds"] = rd + 1
    df = _rs_pd.DataFrame({f: tot[f] / cnt[f] for f in tot}).reindex(ids)
    return df, info


try:
    _rs_t0 = _rs_time.time()
    _rs_ids = _rs_test_ids()
    assert len(set(_rs_ids)) == len(_rs_ids) > 0
    print(f"[region 2/3] {len(_rs_ids)} studies | chunks of {_RS_SPEC['chunk']} | notebook at {(_rs_t0 - _RS_NB_T0) / 3600:.2f} h | "
          f"limit {30240.0 / 3600:.1f} h", flush=True)
    _rs_os.makedirs(_RS_W, exist_ok=True)
    open(_RS_W + "/dense512_build.py", "w").write('#!/usr/bin/env python3\n"""Dense 80-slot native-resolution cache (Kaggle CPU, one shard per kernel).\n\nSame series picker, slice picker, normalisation and 140 mm crop as dense80 (so slot j here is the\nsame slice as slot j there), at IMG=512 (0.27 mm/px ~ native). Adds per-slot DICOM geometry for the\nstage-1 localiser (maps a pixel of any slot to patient mm, so regions transfer between planes).\nOutputs in /kaggle/working: vols_s{SHARD}.npy (n,80,512,512) uint8, masks_s{SHARD}.npy,\ngeom_s{SHARD}.npy (n,80,GEOM) float32, ids_s{SHARD}.npy, meta_s{SHARD}.json.\ngeom columns: ps_row, ps_col, ipp xyz, iop row xyz, iop col xyz, y0, x0, cpx (crop in native px),\nnative rows, native cols, slice thickness, pick index, series length, has_geometry.\n"""\nimport glob\nimport json\nimport os\nimport time\nfrom multiprocessing import Pool\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nfrom pydicom.pixel_data_handlers.util import apply_modality_lut\n\nSHARD = 0\nN_SHARDS = 1\nIMG = 512\nGEOM = 20\nCROP_MM = 140.0\nSPAN_LO, SPAN_HI = 0.02, 0.98\nSLOTS = [("Sagittal", 1, 22), ("Sagittal", 0, 18), ("Coronal", 1, 15),\n         ("Coronal", 0, 10), ("Axial", -1, 15)]\nMAXS = sum(s[2] for s in SLOTS)\nTIME_BUDGET_S = 8.3 * 3600\nOUT = os.environ["RS_CORPUS"]\nSPLIT = os.environ.get("RS_SPLIT", "test")\ncv2.setNumThreads(1)\n\n\ndef find_root():\n    for c in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",\n              "/kaggle/input/rsna-knee-abnormality-detection"):\n        if os.path.exists(c + "/train.csv"):\n            return c\n    for d, _, f in os.walk("/kaggle/input"):\n        if "train_series.csv" in f:\n            return d\n    raise SystemExit("competition data not found")\n\n\ndef order_and_meta(sdir):\n    fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []; info = {}\n    for f in fs:\n        try:\n            h = pydicom.dcmread(f, stop_before_pixels=True)\n            iop = getattr(h, "ImageOrientationPatient", None)\n            ipp = getattr(h, "ImagePositionPatient", None)\n            g = np.zeros(GEOM, np.float32)\n            if iop is not None and ipp is not None and len(iop) == 6:\n                r = np.array(iop[:3], float); c = np.array(iop[3:], float)\n                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))\n                g[2:5] = np.array(ipp, float); g[5:8] = r; g[8:11] = c; g[19] = 1.0\n            else:\n                pos = float(getattr(h, "InstanceNumber", 0) or 0)\n            psv = getattr(h, "PixelSpacing", None); ps = float(psv[0]) if psv is not None else 0.5\n            g[0] = ps; g[1] = float(psv[1]) if psv is not None and len(psv) > 1 else ps\n            try:\n                g[16] = float(getattr(h, "SliceThickness", 0) or 0)\n            except Exception:\n                pass\n            if not info:\n                info = {k: str(getattr(h, k, "") or "") for k in\n                        ("Laterality", "ImageLaterality", "SeriesDescription", "BodyPartExamined",\n                         "PatientPosition", "Manufacturer")}\n            ps_list.append(ps); recs.append((pos, f, ps, g))\n        except Exception:\n            recs.append((0.0, f, 0.5, np.zeros(GEOM, np.float32)))\n    recs.sort(key=lambda x: x[0])\n    med_ps = float(np.median(ps_list)) if ps_list else 0.5\n    return [(f, ps, g) for _, f, ps, g in recs], med_ps, info\n\n\ndef read_px(f):\n    d = pydicom.dcmread(f)\n    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        a = a.max() - a\n    return a\n\n\ndef mm_crop_resize(a, ps):\n    h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))\n    cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2\n    a = a[y0:y0 + cpx, x0:x0 + cpx]\n    interp = cv2.INTER_AREA if cpx >= IMG else cv2.INTER_CUBIC\n    return np.clip(cv2.resize(a, (IMG, IMG), interpolation=interp), 0, 1), (y0, x0, cpx, h, w)\n\n\ndef pick_series(rows, plane, fluid, used):\n    cands = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\ndef fill(vol, geom, offset, picks, cache, files, med_ps):\n    arrays, spacings = [], []\n    for p in picks:\n        p = min(int(p), len(files) - 1)\n        arrays.append(cache.get(p)); ps = files[p][1]; spacings.append(ps if ps > 0 else med_ps)\n    valid = [a for a in arrays if a is not None]\n    lo, hi = (np.percentile(np.concatenate([a.ravel() for a in valid]), [2.0, 98.0])\n              if valid else (0.0, 1.0))\n    for j, (a, ps) in enumerate(zip(arrays, spacings)):\n        if offset + j >= MAXS:\n            break\n        if a is None:\n            continue\n        x, (y0, x0, cpx, h, w) = mm_crop_resize(np.clip((a - lo) / (hi - lo + 1e-6), 0, 1), ps)\n        vol[offset + j] = (x * 255).astype(np.uint8)\n        p = min(int(picks[j]), len(files) - 1)\n        g = files[p][2].copy()\n        g[11:16] = (y0, x0, cpx, h, w); g[17] = p; g[18] = len(files)\n        geom[offset + j] = g\n\n\ndef build_study(args):\n    sid, rows, sdir = args\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    geom = np.zeros((MAXS, GEOM), np.float32)\n    used, offset, infos = set(), 0, []\n    try:\n        for plane, fluid, count in SLOTS:\n            rec = pick_series(rows, plane, fluid, used)\n            if rec is None:\n                offset += count; continue\n            used.add(rec["SeriesInstanceUID"])\n            files, med_ps, info = order_and_meta(f"{sdir}/{sid}/{rec[\'SeriesInstanceUID\']}")\n            info.update(slot=offset, plane=plane, fluid=fluid, n=len(files),\n                        series=rec["SeriesInstanceUID"])\n            infos.append(info)\n            if not files:\n                offset += count; continue\n            n = len(files)\n            lo = int(n * SPAN_LO); hi = max(int(n * SPAN_HI) - 1, lo)\n            picks = (np.linspace(lo, hi, count).round().astype(int) if n > 1\n                     else np.zeros(count, dtype=int))\n            cache = {}\n            for p in sorted(set(picks.tolist())):\n                p = min(int(p), n - 1)\n                try:\n                    cache[p] = read_px(files[p][0])\n                except Exception:\n                    cache[p] = None\n            fill(vol, geom, offset, picks, cache, files, med_ps)\n            offset += count\n            if offset >= MAXS:\n                break\n        err = ""\n    except Exception as e:\n        err = f"{type(e).__name__}: {e}"\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return vol, mask, geom, infos, err\n\n\ndef main():\n    t0 = time.time()\n    root = find_root()\n    sdir = root + f"/{SPLIT}_series"\n    if not os.path.isdir(sdir):\n        sdir = root + f"/{SPLIT}_images"\n    ids = pd.read_csv(root + f"/{SPLIT}.csv")["StudyInstanceUID"].astype(str).tolist()\n    if os.environ.get("RS_IDS"):\n        ids = json.load(open(os.environ["RS_IDS"]))\n    ser = pd.read_csv(root + f"/{SPLIT}_series.csv")\n    ser["StudyInstanceUID"] = ser["StudyInstanceUID"].astype(str)\n    ser["SeriesInstanceUID"] = ser["SeriesInstanceUID"].astype(str)\n    SER = {k: v.to_dict("records") for k, v in ser.groupby("StudyInstanceUID")}\n    b = np.linspace(0, len(ids), N_SHARDS + 1).round().astype(int)\n    mine = ids[b[SHARD]:b[SHARD + 1]]\n    print(f"root {root} | series dir {sdir} | shard {SHARD}/{N_SHARDS}: {len(mine)} studies | "\n          f"cpus {os.cpu_count()}", flush=True)\n\n    vols = np.lib.format.open_memmap(f"{OUT}/vols_s{SHARD}.npy", mode="w+", dtype=np.uint8,\n                                     shape=(len(mine), MAXS, IMG, IMG))\n    masks = np.zeros((len(mine), MAXS), np.uint8)\n    geoms = np.zeros((len(mine), MAXS, GEOM), np.float32)\n    errors, done, series_info = {}, 0, {}\n    jobs = [(sid, SER.get(sid, []), sdir) for sid in mine]\n    with Pool(os.cpu_count()) as pool:\n        for i, (vol, mask, geom, infos, err) in enumerate(pool.imap(build_study, jobs, chunksize=2)):\n            vols[i] = vol; masks[i] = mask; geoms[i] = geom; series_info[mine[i]] = infos; done = i + 1\n            if err:\n                errors[mine[i]] = err\n            if done % 50 == 0 or done == len(mine):\n                el = time.time() - t0\n                print(f"{done}/{len(mine)} | {el/60:.1f} min | eta {el/done*(len(mine)-done)/60:.1f} min "\n                      f"| mean slots {masks[:done].sum(1).mean():.1f} | errors {len(errors)}", flush=True)\n            if time.time() - t0 > TIME_BUDGET_S:\n                print("time budget hit; writing partial shard", flush=True)\n                break\n    vols.flush()\n    np.save(f"{OUT}/masks_s{SHARD}.npy", masks)\n    np.save(f"{OUT}/geom_s{SHARD}.npy", geoms)\n    json.dump(series_info, open(f"{OUT}/series_s{SHARD}.json", "w"))\n    np.save(f"{OUT}/ids_s{SHARD}.npy", np.array(mine))\n    json.dump({"shard": SHARD, "n_shards": N_SHARDS, "n": len(mine), "done": done,\n               "slots": SLOTS, "span": [SPAN_LO, SPAN_HI], "img": IMG, "crop_mm": CROP_MM,\n               "errors": errors, "empty_studies": int((masks[:done].sum(1) == 0).sum()),\n               "minutes": round((time.time() - t0) / 60, 1)},\n              open(f"{OUT}/meta_s{SHARD}.json", "w"), indent=1)\n    print("DONE", json.dumps({"done": done, "errors": len(errors)}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
    open(_RS_W + "/region_spec.py", "w").write('#!/usr/bin/env python3\n"""Native-resolution region specialists on the dense512 cache (PLAN_TO_TOP 2026-10-03).\n\nStage 1 (localiser, no training): the three stacks are prescribed on the joint, so the joint centre is\n    P = (x of the sagittal stack centre, y of the coronal stack centre, z of the axial stack centre)\nin patient mm (LPS), read from the per-slot DICOM geometry in geom_s*.npy. Knee side (which patient\ndirection is lateral) = DICOM Laterality, else the "side" region model (arm `--region side`), else\nsign(P.x). Canonical knee axes: A = lateral, B = anterior, C = superior.\n\nStage 2 (specialists): a region is a set of tile groups; every tile is a 2.5D window (3 adjacent slots)\nwarped from the 512 px slice to a box given in canonical mm around P, at ~native resolution, mirrored\nso that all knees look alike. `tf` sees ONE tibiofemoral compartment per sample: every study gives a\nlateral and a (mirrored) medial sample with the meniscus / OA labels of that compartment.\nTiles -> shared encoder -> tokens (+group, +slice-position) -> transformer mix -> one query per output.\n\nProtocol = family_a.py (folds_nonexpert_5, dev-29 at gold weight, holdout-29 never trained, EMA + last\nmembers, no epoch selection). Outputs: raptor_rs_<tag>.pt/_last.pt, raptor_oof_<tag>_fold<f>.npz and\nraptor_holdout_<tag>.npz (pred has all 12 columns, NaN where the region has no output),\nraptor_ft_<tag>.json; `--region side` writes raptor_side_<tag>.npz (ids, p_left) for every study.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\ncv2.setNumThreads(0)\nHERE = os.path.dirname(os.path.abspath(__file__))\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n       "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nL = {f: i for i, f in enumerate(LAB)}\nWEAK5 = ["Lateral Meniscus", "Lateral OA", "PF OA", "Synovitis", "Contusion"]\nBLOCKS = {"SagF": (0, 22, "sag"), "SagNF": (22, 40, "sag"), "CorF": (40, 55, "cor"),\n          "CorNF": (55, 65, "cor"), "Ax": (65, 80, "ax")}\nIMG = 512\nN_POS = 8\nPARTNER = {"SagF": "SagNF", "CorF": "CorNF"}     # --chan seqfuse: fluid-sensitive block -> its non-fluid partner\nAUX_FINDS = ("Medial OA", "Lateral OA", "PF OA")  # --aux_grades: findings with a report severity grade (0-4) as side target\nAUX_THR = (2, 3, 4)                               # ordinal side outputs: grade >= 2 (any), >= 3 (moderate), >= 4 (severe)\n\n# In-plane image axes as canonical (A lateral, B anterior, C superior) unit vectors:\n# coronal: right = lateral, down = inferior | sagittal: right = posterior, down = inferior |\n# axial: right = lateral, down = posterior.  NORMAL = canonical axis that orders the slices.\nAXES = {"cor": ((1, 0, 0), (0, 0, -1)), "sag": ((0, -1, 0), (0, 0, -1)), "ax": ((1, 0, 0), (0, -1, 0))}\nNORMAL = {"cor": (0, 1, 0), "sag": (1, 0, 0), "ax": (0, 0, 1)}\n\n\ndef grp(block, hbox, vbox, out_wh, sel, n_train, n_eval):\n    """hbox/vbox: mm along the image axes (AXES) relative to P; sel: mm range along NORMAL (or None)."""\n    return dict(block=block, hbox=hbox, vbox=vbox, out=out_wh, sel=sel, n=(n_train, n_eval))\n\n\nREGIONS = {\n    # one tibiofemoral compartment; variant = (compartment sign along A, findings)\n    "tf": dict(\n        variants=[(+1, ["Lateral Meniscus", "Lateral OA"]), (-1, ["Medial Meniscus", "Medial OA"])],\n        groups=[grp("CorF", (-12, 56), (-40, 40), (256, 304), (-35, 35), 5, 8),\n                grp("CorNF", (-12, 56), (-40, 40), (256, 304), (-35, 35), 3, 5),\n                grp("SagF", (-52, 52), (-40, 40), (384, 304), (4, 52), 4, 6),\n                grp("SagNF", (-52, 52), (-40, 40), (384, 304), (4, 52), 3, 5)]),\n    "pf": dict(\n        variants=[(+1, ["PF OA"])],\n        groups=[grp("Ax", (-50, 50), (-82, 6), (368, 320), (-15, 50), 5, 8),\n                grp("SagF", (-86, 0), (-60, 40), (304, 352), (-26, 26), 4, 6),\n                grp("SagNF", (-86, 0), (-60, 40), (304, 352), (-26, 26), 3, 5)]),\n    "rec": dict(\n        variants=[(+1, ["Effusion", "Synovitis", "Baker\'s"])],\n        groups=[grp("Ax", (-70, 70), (-70, 70), (448, 448), None, 5, 8),\n                grp("SagF", (-70, 70), (-75, 45), (448, 384), None, 5, 8),\n                grp("SagNF", (-70, 70), (-75, 45), (448, 384), None, 2, 4)]),\n    "bm": dict(\n        variants=[(+1, ["Contusion", "Fracture"])],\n        groups=[grp("CorF", (-62, 62), (-55, 55), (448, 400), None, 5, 8),\n                grp("SagF", (-62, 62), (-55, 55), (448, 400), None, 5, 8),\n                grp("Ax", (-62, 62), (-62, 62), (400, 400), None, 3, 5),\n                grp("CorNF", (-62, 62), (-55, 55), (448, 400), None, 3, 5)]),\n    # knee side: coronal field incl. the fibula in PATIENT orientation (A := +x), target = left knee\n    "side": dict(\n        variants=[(+1, ["__left__"])],\n        groups=[grp("CorF", (-70, 70), (-45, 70), (224, 184), None, 3, 4),\n                grp("CorNF", (-70, 70), (-45, 70), (224, 184), None, 2, 3)]),\n}\n\n\n# ------------------------------------------------------------------ corpus + localiser\ndef _find(name, roots=(HERE, "/workspace/bundle", "/workspace/out", "/kaggle/input", "/workspace/input")):\n    for root in roots:\n        if not os.path.isdir(root):\n            continue\n        for d, _, fs in os.walk(root):\n            if name in fs:\n                return os.path.join(d, name)\n    return None\n\n\ndef slice_centres(g):\n    """(...,GEOM) -> (...,3) patient mm of the slice centres."""\n    return (g[..., 2:5] + g[..., 5:8] * (g[..., 1:2] * g[..., 15:16] / 2)\n            + g[..., 8:11] * (g[..., 0:1] * g[..., 14:15] / 2))\n\n\nclass Corpus:\n    """Memory-mapped dense512 shards + per-slot geometry; joint centre P and tag laterality per study."""\n\n    def __init__(self, n_shards=8, root=None):\n        self.n_shards, self.root, self.vols = n_shards, root, None\n        ids, masks, geom, tag = [], [], [], {}\n        for i in range(n_shards):\n            ids.append(np.load(self._p(f"ids_s{i}.npy"), allow_pickle=True).astype(str))\n            masks.append(np.load(self._p(f"masks_s{i}.npy")))\n            geom.append(np.load(self._p(f"geom_s{i}.npy")))\n            for u, infos in json.load(open(self._p(f"series_s{i}.json"))).items():\n                s = {x.get("Laterality", "")[:1].upper() for x in infos} & {"L", "R"}\n                if len(s) == 1:\n                    tag[str(u)] = 1.0 if s.pop() == "L" else 0.0\n        self.ids = np.concatenate(ids)\n        self.masks = np.concatenate(masks).astype(bool)\n        self.geom = np.concatenate(geom).astype(np.float64)\n        self.offsets = np.cumsum([0] + [len(x) for x in ids])\n        self.row = {u: i for i, u in enumerate(self.ids)}\n        self.tag_left = tag\n        cen = slice_centres(self.geom)\n        self.P = np.zeros((len(self.ids), 3))\n        for r in range(len(self.ids)):\n            allc = cen[r][self.masks[r]]\n            fall = allc.mean(0) if len(allc) else np.zeros(3)\n            for ax, blocks in enumerate((("SagF", "SagNF"), ("CorF", "CorNF"), ("Ax",))):\n                k = np.concatenate([np.arange(*BLOCKS[b][:2]) for b in blocks])\n                k = k[self.masks[r, k]]\n                self.P[r, ax] = cen[r, k, ax].mean() if len(k) else fall[ax]\n\n    def _p(self, name):\n        p = os.path.join(self.root, name) if self.root else _find(name)\n        if not p or not os.path.exists(p):\n            raise SystemExit(f"dense512 file {name} not found (DENSE_NAME=dense512 fetch_dense80.py)")\n        return p\n\n    def study(self, row):\n        if self.vols is None:\n            self.vols = [np.load(self._p(f"vols_s{i}.npy"), mmap_mode="r") for i in range(self.n_shards)]\n        s = int(np.searchsorted(self.offsets, row, side="right") - 1)\n        return self.vols[s][row - self.offsets[s]]\n\n    def left_prob(self, side_file=None):\n        """P(left knee) per study: Laterality tag > side model > sign of P.x."""\n        p = {u: (1.0 if self.P[i, 0] > 0 else 0.0) for u, i in self.row.items()}\n        src = dict.fromkeys(p, "x")\n        if side_file:\n            z = np.load(side_file, allow_pickle=True)\n            for u, v in zip(z["ids"].astype(str), z["p_left"]):\n                p[u] = float(v); src[u] = "model"\n        for u, v in self.tag_left.items():\n            p[u] = v; src[u] = "tag"\n        return p, src\n\n\ndef to_pixel(g, X):\n    """Patient mm (k,3) -> (u,v) px in the 512 slice with geometry row g."""\n    d = X - g[2:5]\n    x = d @ g[5:8] / g[1]\n    y = d @ g[8:11] / g[0]\n    return np.stack([(x - g[12]) * IMG / g[13], (y - g[11]) * IMG / g[13]], 1)\n\n\n# ------------------------------------------------------------------ data\nclass RegionTiles(Dataset):\n    """Item = (study, variant) -> {group: (n,3,H,W)} tiles + pos/valid vectors, target, weight."""\n\n    def __init__(self, corpus, region, ids, labels, weights, left, train, aug=1.0, eval_phase=0.0,\n                 side_mode=False, chan="adj", aux=None, variants=None):\n        self.c, self.reg, self.train = corpus, REGIONS[region], train\n        self.aux = aux        # {uid: (targets (12, 3), weight (12,))} from load_aux, or None: 3 extra targets per item\n        self.chan = chan      # "seqfuse": tiles of a fluid-sensitive group = [fluid, non-fluid, difference] (see PARTNER)\n        self.labels, self.weights, self.left = labels, weights, left\n        self.items = [(u, v) for u in ids for v in (variants or range(len(self.reg["variants"])))]\n        self.aug = aug if train else 0.0\n        self.eval_phase, self.side_mode = eval_phase, side_mode\n\n    def __len__(self):\n        return len(self.items)\n\n    def _pick(self, cand, n):\n        if self.train:\n            return np.sort(np.random.choice(cand, n, replace=len(cand) < n))\n        frac = (np.linspace(0.0, 1.0, n) + self.eval_phase) % 1.0 if n > 1 else np.array([0.5])\n        return cand[np.round(frac * (len(cand) - 1)).astype(int)]\n\n    def __getitem__(self, i):\n        uid, vi = self.items[i]\n        sgn, finds = self.reg["variants"][vi]\n        row = self.c.row[uid]\n        vol, msk, geom, P = self.c.study(row), self.c.masks[row], self.c.geom[row], self.c.P[row].copy()\n        if self.side_mode:       # patient orientation; random mirror flips the target\n            lat = random.choice((1.0, -1.0)) if self.train else 1.0\n            t = self.labels.get(uid, 0.0)                 # untagged studies are only predicted\n            y = np.array([t if lat > 0 else 1.0 - t], np.float32)\n            w = np.array([self.weights.get(uid, 0.0)], np.float32)\n        else:                    # A points lateral for the lateral variant, medial for the mirrored one\n            lat = (1.0 if self.left[uid] >= 0.5 else -1.0) * sgn\n            idx = [L[f] for f in finds]\n            y, w = self.labels[uid][idx].astype(np.float32), self.weights[uid][idx].astype(np.float32)\n            if self.aux is not None:     # severity grade of this variant\'s OA finding; weight 0 = not mentioned / no report\n                k = next(L[f] for f in finds if f in AUX_FINDS)\n                ay, aw = self.aux.get(uid, (np.zeros((len(LAB), len(AUX_THR)), np.float32), np.zeros(len(LAB), np.float32)))\n                y = np.concatenate([y, ay[k]])\n                w = np.concatenate([w, np.full(len(AUX_THR), aw[k], np.float32)])\n        E = np.array([[lat, 0, 0], [0, -1, 0], [0, 0, 1]], float)   # canonical A,B,C in patient LPS\n        rot = math.radians(random.uniform(-8, 8)) * self.aug\n        sc = 1.0 + random.uniform(-0.08, 0.08) * self.aug\n        P = P + np.random.uniform(-5, 5, 3) * self.aug\n        cr, sr = math.cos(rot), math.sin(rot)\n        out = {}\n        for gi, gdef in enumerate(self.reg["groups"]):\n            lo, hi, plane = BLOCKS[gdef["block"]]\n            n = gdef["n"][0 if self.train else 1]\n            W, H = gdef["out"]\n            eh, ev = (np.array(a, float) @ E for a in AXES[plane])\n            nrm = np.array(NORMAL[plane], float) @ E\n            valid = np.where(msk[lo:hi])[0]\n            tiles = np.zeros((n, 3, H, W), np.float32)\n            pos = np.zeros(n, np.int64)\n            ok = 0\n            if len(valid) >= 3:\n                coord = (slice_centres(geom[lo:hi]) - P) @ nrm\n                cand = valid[(valid > valid.min()) & (valid < valid.max())]\n                rng = gdef["sel"] or (coord[valid].min(), coord[valid].max())\n                inr = cand[(coord[cand] >= rng[0]) & (coord[cand] <= rng[1])]\n                if len(inr) == 0:\n                    inr = cand[np.argsort(np.abs(coord[cand] - np.mean(rng)))[:1]]\n                order = inr[np.argsort(coord[inr])]\n                ok = 1\n                (h0, h1), (v0, v1) = gdef["hbox"], gdef["vbox"]\n                hc, vc, hh, vh = (h0 + h1) / 2, (v0 + v1) / 2, (h1 - h0) / 2 * sc, (v1 - v0) / 2 * sc\n                dst = np.float32([[0, 0], [W, 0], [0, H]])\n                # --chan seqfuse: the non-fluid series of the same plane, matched in PATIENT mm (each slice is warped\n                # with its own geometry, so the two sequences are registered in the tile)\n                plo, phi = BLOCKS[PARTNER[gdef["block"]]][:2] if (self.chan == "seqfuse" and gdef["block"] in PARTNER) else (0, 0)\n                pvalid = np.where(msk[plo:phi])[0]\n                pcoord = (slice_centres(geom[plo:phi]) - P) @ nrm if len(pvalid) else None\n                for t, c in enumerate(self._pick(order, n)):\n                    c = int(c)\n                    pos[t] = int(np.clip((coord[c] - rng[0]) / max(rng[1] - rng[0], 1e-3) * N_POS, 0, N_POS - 1))\n                    corners = []\n                    for a, b in ((-hh, -vh), (hh, -vh), (-hh, vh)):\n                        a, b = a * cr - b * sr, a * sr + b * cr\n                        corners.append(P + eh * (hc + a) + ev * (vc + b))\n                    corners = np.array(corners)\n                    M = cv2.getAffineTransform(to_pixel(geom[lo + c], corners).astype(np.float32), dst)\n                    if len(pvalid):\n                        k2 = int(pvalid[np.argmin(np.abs(pcoord[pvalid] - coord[c]))])\n                        M2 = cv2.getAffineTransform(to_pixel(geom[plo + k2], corners).astype(np.float32), dst)\n                        fl = cv2.warpAffine(np.ascontiguousarray(vol[lo + c]), M, (W, H), flags=cv2.INTER_LINEAR) / 255.0\n                        nf = cv2.warpAffine(np.ascontiguousarray(vol[plo + k2]), M2, (W, H), flags=cv2.INTER_LINEAR) / 255.0\n                        tiles[t, 0], tiles[t, 1], tiles[t, 2] = fl, nf, np.clip(0.5 + (fl - nf) / 2, 0, 1)\n                        continue\n                    # neighbours ordered along the canonical normal so mirrored knees stay consistent\n                    nb = (c - 1, c, c + 1) if coord[c + 1] >= coord[c - 1] else (c + 1, c, c - 1)\n                    for ch, k in enumerate(nb):\n                        tiles[t, ch] = cv2.warpAffine(np.ascontiguousarray(vol[lo + k]), M, (W, H),\n                                                      flags=cv2.INTER_LINEAR) / 255.0\n            if self.aug:\n                tiles = np.clip(tiles * (1.0 + (random.random() - 0.5) * 0.25) + (random.random() - 0.5) * 0.06, 0, 1)\n            out[f"x{gi}"] = torch.from_numpy(tiles)\n            out[f"p{gi}"] = torch.from_numpy(pos)\n            out[f"v{gi}"] = torch.tensor(ok)\n        out["var"] = torch.tensor(vi)\n        return out, torch.from_numpy(y), torch.from_numpy(w)\n\n\ndef collate(batch):\n    x = {k: torch.stack([b[0][k] for b in batch]) for k in batch[0][0]}\n    return x, torch.stack([b[1] for b in batch]), torch.stack([b[2] for b in batch])\n\n\n# ------------------------------------------------------------------ model\nclass RegionNet(nn.Module):\n    def __init__(self, arch, n_groups, n_var, n_out, pretrained=True, d=384, heads=6, mix_layers=2,\n                 drop=0.1, grad_ckpt=False):\n        super().__init__()\n        import timm\n        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0, global_pool="avg")\n        if grad_ckpt:\n            self.backbone.set_grad_checkpointing(True)\n        cfg = self.backbone.pretrained_cfg\n        self.register_buffer("mean", torch.tensor(cfg.get("mean", (0.485, 0.456, 0.406))).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor(cfg.get("std", (0.229, 0.224, 0.225))).view(1, 3, 1, 1))\n        self.proj = nn.Sequential(nn.Linear(self.backbone.num_features, d), nn.LayerNorm(d))\n        self.gemb = nn.Embedding(n_groups, d)\n        self.pemb = nn.Embedding(N_POS, d)\n        self.vemb = nn.Embedding(n_var, d)\n        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, batch_first=True, norm_first=True)\n        self.mix = nn.TransformerEncoder(layer, mix_layers, enable_nested_tensor=False)\n        self.q = nn.Parameter(torch.randn(n_out, d) * 0.02)\n        self.xattn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)\n        self.norm = nn.LayerNorm(d)\n        self.cls_w = nn.Parameter(torch.randn(n_out, d) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        self.n_groups = n_groups\n\n    def forward(self, x):\n        toks, valid = [], []\n        for g in range(self.n_groups):\n            t = x[f"x{g}"]\n            B, N = t.shape[:2]\n            f = self.backbone((t.flatten(0, 1) - self.mean) / self.std).view(B, N, -1)\n            toks.append(self.proj(f.float()) + self.gemb.weight[g] + self.pemb(x[f"p{g}"]))\n            valid.append(x[f"v{g}"].bool()[:, None].expand(B, N))\n        h, valid = torch.cat(toks, 1), torch.cat(valid, 1)\n        valid = valid | (~valid.any(1, keepdim=True))\n        h = self.mix(h + self.vemb(x["var"])[:, None], src_key_padding_mask=~valid)\n        q = self.q[None].expand(h.shape[0], -1, -1) + self.vemb(x["var"])[:, None]\n        o, _ = self.xattn(q, h, h, key_padding_mask=~valid)\n        o = self.norm(o + q)\n        return (o * self.cls_w).sum(-1) + self.cls_b\n\n\n# ------------------------------------------------------------------ labels\ndef load_targets(a, corpus):\n    tr = pd.read_csv(a.train_csv or _find("train.csv"))\n    tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)\n    gold = tr[tr[LAB].notna().all(1)].set_index("StudyInstanceUID")[LAB].astype(np.float32)\n    hold = set(json.load(open(a.holdout)))\n    lab = pd.read_parquet(a.labels) if a.labels.endswith(".parquet") else pd.read_csv(a.labels)\n    lab["StudyInstanceUID"] = lab["StudyInstanceUID"].astype(str)\n    lab = lab.set_index("StudyInstanceUID")[LAB].astype(np.float32)\n    have = set(corpus.ids)\n    labels, weights = {}, {}\n    for u, r in lab.iterrows():\n        if u in have and u not in gold.index:\n            labels[u] = r.values\n            weights[u] = 0.25 + 0.75 * np.clip(np.abs(2 * r.values - 1), 0, 1)\n    for u, r in gold.iterrows():\n        if u in have:\n            labels[u] = r.values\n            weights[u] = np.full(len(LAB), a.gold_w, np.float32)\n    holdout = [u for u in gold.index if u in hold and u in have]\n    dev = [u for u in gold.index if u not in hold and u in have]\n    folds = json.load(open(a.fold_file))["folds"]\n    nonexp = [u for u in lab.index if u in have and u not in gold.index]\n    oof = [u for u in nonexp if folds.get(u, -1) == a.fold] if a.fold >= 0 else []\n    oofset = set(oof)\n    return labels, weights, [u for u in nonexp if u not in oofset] + dev, oof, holdout\n\n\ndef load_aux(path, w):\n    """Report severity grades (scripts/graded_labels.py: 0 explicitly normal, 1 not mentioned, 2 low-grade, 3 moderate,\n    4 severe) -> {uid: (ordinal targets (12, 3) = grade >= AUX_THR, weight (12,))}. A finding that the report does not\n    mention is unknown (weight 0), not negative. `w` is the summed weight of the three side outputs of a known finding."""\n    g = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)\n    g["StudyInstanceUID"] = g["StudyInstanceUID"].astype(str)\n    G = g.set_index("StudyInstanceUID")[LAB].to_numpy(np.float32)\n    thr = np.array(AUX_THR, np.float32)\n    out = {}\n    for u, r in zip(g["StudyInstanceUID"], G):\n        known = np.isfinite(r) & (r != 1)\n        out[u] = ((np.nan_to_num(r)[:, None] >= thr).astype(np.float32), known.astype(np.float32) * w / len(AUX_THR))\n    return out\n\n\ndef auc(y, p):\n    from sklearn.metrics import roc_auc_score\n    y = (np.asarray(y) >= 0.5).astype(int)\n    return float(roc_auc_score(y, p)) if 0 < y.sum() < len(y) else float("nan")\n\n\n# ------------------------------------------------------------------ main\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--labels", required=True)\n    ap.add_argument("--region", required=True, choices=sorted(REGIONS))\n    ap.add_argument("--train_csv", default=None)\n    ap.add_argument("--fold_file", default=None)\n    ap.add_argument("--holdout", default=None)\n    ap.add_argument("--side_file", default=None, help="raptor_side_*.npz (default: search)")\n    ap.add_argument("--fold", type=int, default=0, help="-1 = train on all non-holdout studies")\n    ap.add_argument("--arch", default="convnext_small.fb_in22k_ft_in1k")\n    ap.add_argument("--epochs", type=int, default=10)\n    ap.add_argument("--bs", type=int, default=6)\n    ap.add_argument("--accum", type=int, default=2)\n    ap.add_argument("--bb_lr", type=float, default=6e-5)\n    ap.add_argument("--head_lr", type=float, default=6e-4)\n    ap.add_argument("--wd", type=float, default=0.05)\n    ap.add_argument("--warmup", type=float, default=0.06)\n    ap.add_argument("--ema", type=float, default=0.999)\n    ap.add_argument("--gold_w", type=float, default=3.0)\n    ap.add_argument("--eval_passes", type=int, default=2)\n    ap.add_argument("--eval_every", type=int, default=2)\n    ap.add_argument("--workers", type=int, default=8)\n    ap.add_argument("--grad_ckpt", action="store_true")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--tag", default="rs")\n    ap.add_argument("--out", default=HERE)\n    ap.add_argument("--data_root", default=None)\n    ap.add_argument("--n_shards", type=int, default=8)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--no_pretrained", action="store_true")\n    ap.add_argument("--smoke", action="store_true")\n    ap.add_argument("--dump", default=None, help="write a jpg of the first samples\' tiles and exit")\n    ap.add_argument("--chan", default="adj", choices=["adj", "seqfuse"],\n                    help="tile channels: adj = 3 adjacent slices; seqfuse = fluid-sensitive slice, the non-fluid slice of the "\n                         "same plane nearest in patient mm, and their difference (fluid-sensitive groups only)")\n    ap.add_argument("--aux_grades", default=None,\n                    help="report severity grades (labels_graded_v1.parquet): three extra ordinal outputs for the OA finding "\n                         "of each variant (tf, pf), trained with a masked loss and ignored at test time")\n    ap.add_argument("--aux_w", type=float, default=0.5, help="summed loss weight of the three side outputs")\n    ap.add_argument("--predict", default=None,\n                    help="comma list of raptor_rs_*.pt checkpoints of ONE region (paths or names under the search "\n                         "roots): score every study of the corpus without labels, mean over the checkpoints, and write "\n                         "raptor_rspred_<tag>.npz (ids, pred (n, 12), NaN outside the region) or, for the side region, "\n                         "raptor_side_<tag>.npz (ids, p_left). Used by the test-time leg (ours_leg_rs.py)")\n    ap.add_argument("--only_findings", default=None,\n                    help="--predict: semicolon list of findings; only the variants that hold one of them are scored "\n                         "(tf: \'Lateral OA\' = the lateral compartment alone, half the cost); the others stay NaN")\n    ap.add_argument("--channels_last", action="store_true", help="--predict: backbone in channels-last memory format")\n    a = ap.parse_args()\n    if a.predict:\n        return predict_only(a)\n    a.fold_file = a.fold_file or _find("folds_nonexpert_5.json")\n    a.holdout = a.holdout or _find("expert_holdout.json")\n    if not a.fold_file or not a.holdout:\n        raise SystemExit("folds_nonexpert_5.json / expert_holdout.json not found")\n    if a.smoke:\n        a.epochs, a.limit, a.eval_passes, a.eval_every = 1, 24, 1, 1\n        a.workers = min(a.workers, 2)\n    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    reg = REGIONS[a.region]\n    side_mode = a.region == "side"\n\n    corpus = Corpus(a.n_shards, a.data_root)\n    labels, weights, train_ids, oof_ids, hold_ids = load_targets(a, corpus)\n    a.side_file = a.side_file or (None if side_mode else _find_side())\n    left, src = corpus.left_prob(a.side_file)\n    cols = [[L.get(f, -1) for f in finds] for _, finds in reg["variants"]]\n    aux = None\n    if a.aux_grades:\n        if not all(any(f in AUX_FINDS for f in finds) for _, finds in reg["variants"]):\n            raise SystemExit(f"--aux_grades: region {a.region} has a variant without an OA finding")\n        path = a.aux_grades if os.path.exists(a.aux_grades) else _find(os.path.basename(a.aux_grades))\n        if not path:\n            raise SystemExit(f"--aux_grades: {a.aux_grades} not found")\n        aux = load_aux(path, a.aux_w)\n        n_known = {f: int(sum(aux[u][1][L[f]] > 0 for u in train_ids if u in aux)) for _, fs in reg["variants"]\n                   for f in fs if f in AUX_FINDS}\n        print(f"aux grades {os.path.basename(path)} | train studies with a known grade {n_known} | weight {a.aux_w}", flush=True)\n    n_aux = len(AUX_THR) if aux is not None else 0\n    if side_mode:                 # target = Laterality tag; only tagged studies train / score\n        tagged = corpus.tag_left\n        labels = {u: tagged[u] for u in tagged}\n        weights = dict.fromkeys(tagged, 1.0)\n        train_ids = [u for u in train_ids if u in tagged]\n        oof_ids = [u for u in oof_ids + hold_ids if u in tagged]\n        hold_ids = []\n    if a.limit:\n        train_ids = train_ids[: a.limit]; oof_ids = oof_ids[: max(8, a.limit // 2)]\n    n_src = {s: sum(v == s for v in src.values()) for s in ("tag", "model", "x")}\n    print(f"device {dev} | region {a.region} | arch {a.arch} | fold {a.fold} | train {len(train_ids)} | "\n          f"oof {len(oof_ids)} | holdout {len(hold_ids)} | side source {n_src}", flush=True)\n\n    mk = lambda ids, train, phase=0.0: RegionTiles(corpus, a.region, ids, labels, weights, left, train,\n                                                    eval_phase=phase, side_mode=side_mode, chan=a.chan, aux=aux)\n    if a.dump:\n        ds = mk((train_ids + oof_ids)[:4], False)\n        rows = []\n        for i in range(len(ds)):\n            x, y, _ = ds[i]\n            for g in range(len(reg["groups"])):\n                t = (x[f"x{g}"][:, 1].numpy() * 255).astype(np.uint8)\n                rows.append(cv2.resize(np.hstack(list(t[:4])), None, fx=0.6, fy=0.6))\n        w = max(r.shape[1] for r in rows)\n        cv2.imwrite(a.dump, np.vstack([np.pad(r, ((0, 4), (0, w - r.shape[1]))) for r in rows]))\n        print("dumped", a.dump, [ds.items[i] for i in range(len(ds))], [left[u] for u, _ in ds.items])\n        return 0\n\n    tl = DataLoader(mk(train_ids, True), batch_size=a.bs, shuffle=True, drop_last=True,\n                    num_workers=a.workers, collate_fn=collate, pin_memory=dev == "cuda",\n                    persistent_workers=a.workers > 0,\n                    worker_init_fn=lambda w: np.random.seed((a.seed * 1000 + w + int(time.time())) % 2**31))\n    n_main = len(cols[0])\n    n_out = n_main + n_aux        # outputs: the region\'s findings, then the ordinal side outputs (training only)\n    model = RegionNet(a.arch, len(reg["groups"]), len(reg["variants"]), n_out,\n                      pretrained=not a.no_pretrained, grad_ckpt=a.grad_ckpt).to(dev)\n    bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]\n    hd = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]\n    opt = torch.optim.AdamW([{"params": bb, "lr": a.bb_lr}, {"params": hd, "lr": a.head_lr}], weight_decay=a.wd)\n    steps = max(1, a.epochs * (len(tl) // a.accum))\n    warm = max(1, int(a.warmup * steps))\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))\n    if side_mode:\n        pw = torch.ones(1, len(cols), n_out, device=dev)\n    else:\n        Y = np.stack([labels[u] for u in train_ids])\n        prev = np.clip(np.stack([Y[:, c].mean(0) for c in cols]), 0.03, 0.7)        # (variants, n_out)\n        pw = torch.tensor(np.clip((1 - prev) / prev, 1, 6), dtype=torch.float32, device=dev)[None]\n        pw = torch.cat([pw, torch.ones(1, len(cols), n_aux, device=dev)], 2)           # side outputs: no class weight\n    ema = {k: v.detach().float().clone() for k, v in model.state_dict().items() if v.is_floating_point()}\n\n    def fwd(x):\n        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}\n        with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == "cuda"):\n            return model(x).float(), x["var"]\n\n    def predict(ids, passes):\n        """-> (len(ids), 12) with NaN for findings outside the region (side: column 0 = p_left)."""\n        model.eval()\n        acc = 0\n        for p in range(passes):\n            ds = mk(ids, False, 0.0 if passes == 1 else p / passes)\n            dl = DataLoader(ds, batch_size=a.bs, shuffle=False, num_workers=a.workers, collate_fn=collate)\n            out = []\n            with torch.no_grad():\n                for x, _, _ in dl:\n                    out.append(torch.sigmoid(fwd(x)[0]).cpu().numpy())\n            acc = acc + np.concatenate(out)\n        acc = (acc / passes).reshape(len(ids), len(cols), n_out)\n        P = np.full((len(ids), len(LAB)), np.nan, np.float32)\n        for v, c in enumerate(cols):\n            for j, k in enumerate(c):\n                P[:, k if k >= 0 else 0] = acc[:, v, j]\n        return P\n\n    def score(P, ids):\n        if side_mode:\n            y = np.array([labels[u] for u in ids])\n            return {"auc": auc(y, P[:, 0]), "acc": float(((P[:, 0] >= 0.5) == (y >= 0.5)).mean())}\n        Yt = np.stack([labels[u] for u in ids])\n        return {LAB[k]: auc(Yt[:, k], P[:, k]) for c in cols for k in c}\n\n    def swap_in(sd):\n        cur = model.state_dict()\n        with torch.no_grad():\n            for k, v in sd.items():\n                cur[k].copy_(v.to(cur[k].dtype))\n\n    hist, t0, step = [], time.time(), 0\n    for ep in range(a.epochs):\n        model.train(); tot = 0.0; n = 0\n        opt.zero_grad(set_to_none=True)\n        for it, (x, y, w) in enumerate(tl):\n            logit, var = fwd(x)\n            y, w = y.to(dev), w.to(dev)\n            per = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pw[0][var], reduction="none") * w\n            loss = per[:, :n_main].mean() + per[:, n_main:].sum(1).mean()   # the finding loss keeps its scale; side term added\n            (loss / a.accum).backward()\n            if (it + 1) % a.accum == 0:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)\n                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1\n                with torch.no_grad():\n                    d = min(a.ema, (1 + step) / (10 + step))\n                    for k, v in model.state_dict().items():\n                        if k in ema:\n                            ema[k].mul_(d).add_(v.detach().float(), alpha=1 - d)\n            tot += loss.item(); n += 1\n            if it % 100 == 0:\n                print(f"  ep{ep} it{it}/{len(tl)} loss {tot / n:.4f} | {time.time() - t0:.0f}s", flush=True)\n        rec = {"ep": ep, "loss": tot / max(n, 1), "sec": round(time.time() - t0)}\n        if oof_ids and ((ep + 1) % a.eval_every == 0 or ep == a.epochs - 1):\n            live = {k: v.detach().clone() for k, v in model.state_dict().items()}\n            swap_in(ema)\n            rec["oof_ema"] = {k: round(v, 4) for k, v in score(predict(oof_ids, 1), oof_ids).items()}\n            swap_in(live)\n        hist.append(rec)\n        print(f"ep{ep} {json.dumps(rec)}", flush=True)\n\n    last = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    meta_ck = {"arch": a.arch, "region": a.region, "spec": reg, "lab": LAB, "fold": a.fold, "seed": a.seed, "chan": a.chan,\n               "n_aux": n_aux}\n    torch.save({"model": last, **meta_ck}, os.path.join(a.out, f"raptor_rs_{a.tag}_last.pt"))\n    res = {"tag": a.tag, "args": vars(a), "hist": hist, "n_train": len(train_ids), "n_oof": len(oof_ids),\n           "n_holdout": len(hold_ids), "members": ["last", "ema"], "side_source": n_src}\n    all_ids = list(corpus.ids) if side_mode else []\n    po, ph, pa = [], [], []\n    for name, sd in (("last", None), ("ema", ema)):\n        if sd is not None:\n            swap_in(sd)\n            torch.save({"model": {k: v.detach().cpu() for k, v in model.state_dict().items()}, **meta_ck},\n                       os.path.join(a.out, f"raptor_rs_{a.tag}.pt"))\n        if oof_ids:\n            po.append(predict(oof_ids, a.eval_passes))\n        if hold_ids:\n            ph.append(predict(hold_ids, a.eval_passes))\n        if all_ids and not a.smoke:\n            pa.append(predict(all_ids, 1))\n    if oof_ids:\n        Po = np.mean(po, 0)\n        res["oof"] = score(Po, oof_ids)\n        if not side_mode:\n            np.savez(os.path.join(a.out, f"raptor_oof_{a.tag}_fold{a.fold}.npz"), pred=Po,\n                     truth=np.stack([labels[u] for u in oof_ids]), ids=np.array(oof_ids), fold=a.fold,\n                     pred_last=po[0], pred_ema=po[1])\n    if hold_ids:\n        Ph = np.mean(ph, 0)\n        res["holdout29"] = score(Ph, hold_ids)\n        np.savez(os.path.join(a.out, f"raptor_holdout_{a.tag}.npz"), pred=Ph,\n                 truth=np.stack([labels[u] for u in hold_ids]), ids=np.array(hold_ids),\n                 pred_last=ph[0], pred_ema=ph[1])\n    if pa:\n        p_left = np.mean(pa, 0)[:, 0]\n        np.savez(os.path.join(a.out, f"raptor_side_{a.tag}.npz"), ids=np.array(all_ids), p_left=p_left)\n        x_left = (corpus.P[:, 0] > 0)\n        res["side_all"] = {"n": len(all_ids), "frac_left": float((p_left >= 0.5).mean()),\n                           "agree_sign_x": float(((p_left >= 0.5) == x_left).mean()),\n                           "unsure": int(((p_left > 0.1) & (p_left < 0.9)).sum())}\n    json.dump(res, open(os.path.join(a.out, f"raptor_ft_{a.tag}.json"), "w"), indent=1, default=str)\n    print("RESULT " + json.dumps({k: res[k] for k in ("oof", "holdout29", "side_all") if k in res}), flush=True)\n    print(f"done in {time.time() - t0:.0f}s", flush=True)\n    return 0\n\n\ndef predict_only(a):\n    """Labels-free scoring of a whole corpus with trained checkpoints (same tiles and passes as training-time predict)."""\n    t0 = time.time()\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    paths = [x if os.path.exists(x) else _find(os.path.basename(x)) for x in a.predict.split(",") if x]\n    if not paths or not all(paths):\n        raise SystemExit(f"--predict: checkpoint not found among {a.predict}")\n    cks = [torch.load(x, map_location="cpu", weights_only=False) for x in paths]\n    region = cks[0]["region"]\n    assert all(c["region"] == region and c["arch"] == cks[0]["arch"] for c in cks), "one region / arch per call"\n    reg, side_mode = REGIONS[region], region == "side"\n    corpus = Corpus(a.n_shards, a.data_root)\n    ids = list(corpus.ids)\n    left, src = corpus.left_prob(None if side_mode else a.side_file)\n    cols = [[L.get(f, -1) for f in finds] for _, finds in reg["variants"]]\n    vsel = list(range(len(cols)))\n    if a.only_findings:\n        want = {f.strip() for f in a.only_findings.split(";") if f.strip()}\n        vsel = [v for v, (_, finds) in enumerate(reg["variants"]) if want & set(finds)]\n        if not vsel:\n            raise SystemExit(f"--only_findings {sorted(want)}: none of them belongs to region {region}")\n    n_out = len(cols[0]) + cks[0].get("n_aux", 0)     # side outputs of --aux_grades models are scored and dropped below\n    assert all(c.get("n_aux", 0) == cks[0].get("n_aux", 0) for c in cks), "checkpoints with and without side outputs: call twice"\n    zero = np.zeros(len(LAB), np.float32)\n    labels, weights = ({}, {}) if side_mode else (dict.fromkeys(ids, zero), dict.fromkeys(ids, zero))\n    model = RegionNet(cks[0]["arch"], len(reg["groups"]), len(reg["variants"]), n_out, pretrained=False).to(dev)\n    model.eval()\n    if a.channels_last:\n        model.backbone.to(memory_format=torch.channels_last)\n        model.backbone.register_forward_pre_hook(lambda m, x: (x[0].contiguous(memory_format=torch.channels_last),))\n    n_src = {k: sum(v == k for v in src.values()) for k in ("tag", "model", "x")}\n    print(f"predict | device {dev} | region {region} | {len(paths)} checkpoint(s) | studies {len(ids)} | "\n          f"passes {a.eval_passes} | variants {vsel} | side source {n_src}", flush=True)\n    acc = 0\n    for ci, ck in enumerate(cks):\n        model.load_state_dict(ck["model"], strict=True)\n        for p in range(a.eval_passes):\n            ds = RegionTiles(corpus, region, ids, labels, weights, left, False,\n                             eval_phase=0.0 if a.eval_passes == 1 else p / a.eval_passes, side_mode=side_mode,\n                             chan=ck.get("chan", "adj"), variants=vsel)\n            dl = DataLoader(ds, batch_size=a.bs, shuffle=False, num_workers=a.workers, collate_fn=collate)\n            out = []\n            with torch.no_grad():\n                for x, _, _ in dl:\n                    x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}\n                    with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):   # T4: no native bfloat16\n                        out.append(torch.sigmoid(model(x).float()).cpu().numpy())\n            acc = acc + np.concatenate(out)\n            print(f"  checkpoint {os.path.basename(paths[ci])} pass {p} | {time.time() - t0:.0f}s", flush=True)\n    acc = (acc / (len(cks) * a.eval_passes)).reshape(len(ids), len(vsel), n_out)\n    P = np.full((len(ids), len(LAB)), np.nan, np.float32)\n    for vi, v in enumerate(vsel):\n        for j, k in enumerate(cols[v]):\n            P[:, k if k >= 0 else 0] = acc[:, vi, j]\n    if side_mode:\n        np.savez(os.path.join(a.out, f"raptor_side_{a.tag}.npz"), ids=np.array(ids), p_left=P[:, 0])\n    else:\n        np.savez(os.path.join(a.out, f"raptor_rspred_{a.tag}.npz"), ids=np.array(ids), pred=P)\n    print(f"predict done | {len(ids)} studies | {time.time() - t0:.0f}s", flush=True)\n    return 0\n\n\ndef _find_side():\n    for root in ("/workspace/out", HERE, "/workspace/bundle"):\n        if os.path.isdir(root):\n            for d, _, fs in os.walk(root):\n                for f in fs:\n                    if f.startswith("raptor_side_") and f.endswith(".npz") and "smoke" not in f:\n                        return os.path.join(d, f)\n    return None\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    _rs_df, _rs_info = _rs_score(_rs_ids, 30240.0)
    _rs_df.index.name = "StudyInstanceUID"; _rs_df.to_csv(_RS_CSV)
    assert _rs_df.notna().all().all(), "a specialist left NaN"
    _rs_ok = True
    print(f"[region 2/3] scored {len(_rs_df)} studies | findings {list(_rs_df.columns)} | rounds {_rs_info['rounds']} | corpus errors "
          f"{_rs_info['errors']}, empty {_rs_info['empty']} | {(_rs_time.time() - _rs_t0) / 60:.1f} min", flush=True)
except Exception as _rs_e:
    print(f"[region 2/3] FAILED ({type(_rs_e).__name__}: {_rs_e})", flush=True)
    if _RS_STRICT:
        raise
for _rs_p in _rs_glob.glob(_RS_W + "/*.py") + _rs_glob.glob(_RS_W + "/__pycache__/*") + _rs_glob.glob(_RS_W + "/side_*.npz") + _rs_glob.glob(_RS_W + "/raptor_rspred_*.npz"):
    _rs_os.remove(_rs_p)


In [ ]:
# ==== REGION 3/3 (alone): every column 0.5 except the specialists' findings ====
assert _rs_ok
_rs_o = _rs_pd.read_csv(_RS_CSV, dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID")
_rs_out = _rs_pd.DataFrame({"StudyInstanceUID": _rs_ids})
for _rs_l in _RS_LAB:
    _rs_out[_rs_l] = 0.5
for _rs_f in _rs_o.columns:
    _rs_c = _rs_o[_rs_f].reindex(_rs_ids)
    assert _rs_c.notna().all(), f"{_rs_f}: studies missing"
    assert len(_rs_ids) <= 10 or float(_rs_c.std()) > 1e-9, f"{_rs_f}: constant column"
    _rs_out[_rs_f] = _rs_c.to_numpy()
assert _rs_np.isfinite(_rs_out[_RS_LAB].to_numpy(float)).all() and _rs_out.columns.tolist() == ["StudyInstanceUID", *_RS_LAB]
_rs_out.to_csv("/kaggle/working/submission.csv", index=False)
print(f"[region 3/3] alone: {list(_rs_o.columns)} from the specialists ({_rs_info['rounds']} round(s)), the other columns 0.5 | "
      f"submission.csv ok {_rs_out.shape} | notebook at {(_rs_time.time() - _RS_NB_T0) / 3600:.2f} h", flush=True)
